# Week 2 - Korean Sentence Embedding + Semantic Retrieval

This week builds sentence embeddings and a dense retrieval pipeline for Korean text.


# Day 1 - Sentence Embedding Baseline

Create a pretrained sentence-embedding baseline on KLUE-STS.


In [1]:
import random
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, set_seed
from sklearn.metrics.pairwise import cosine_similarity

SEEDS = [17, 47, 378]
SEED = 378
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
dataset = load_dataset("klue", "sts")

dataset

README.md: 0.00B [00:00, ?B/s]

sts/train-00000-of-00001.parquet:   0%|          | 0.00/1.52M [00:00<?, ?B/s]

sts/validation-00000-of-00001.parquet:   0%|          | 0.00/68.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/519 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 11668
    })
    validation: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 519
    })
})

In [3]:
model_name = "klue/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

print("Model:", model_name)
print("Device:", device)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: klue/roberta-base
Device: cuda


In [4]:
def get_token_lengths(examples):
    lengths_1 = [
        len(tokenizer.encode(
            text,
            add_special_tokens=True
        ))
        for text in examples["sentence1"]
    ]

    lengths_2 = [
        len(tokenizer.encode(
            text,
            add_special_tokens=True
        ))
        for text in examples["sentence2"]
    ]

    return {
        "length_1": lengths_1,
        "length_2": lengths_2
    }

In [5]:
train_lengths = get_token_lengths(dataset["train"])

val_lengths = get_token_lengths(dataset["validation"])

In [6]:
all_lengths = (
    train_lengths["length_1"]
    + train_lengths["length_2"]
    + val_lengths["length_1"]
    + val_lengths["length_2"]
)

lengths = np.array(all_lengths)

print(f"평균: {lengths.mean():.2f}")
print(f"중앙값: {np.median(lengths):.2f}")
print(f"최대: {lengths.max()}")
print(f"90%: {np.percentile(lengths, 90):.2f}")
print(f"95%: {np.percentile(lengths, 95):.2f}")
print(f"99%: {np.percentile(lengths, 99):.2f}")

평균: 19.80
중앙값: 17.00
최대: 81
90%: 32.00
95%: 39.00
99%: 49.00


In [7]:
for max_length in [32, 64, 128, 256]:
    percentage = (lengths > max_length).mean() * 100

    print(
        f"max_length={max_length}: "
        f"{percentage:.2f}% 초과"
    )

max_length=32: 9.77% 초과
max_length=64: 0.02% 초과
max_length=128: 0.00% 초과
max_length=256: 0.00% 초과


In [8]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask

In [9]:
def get_sentence_embedding(
    sentences,
    max_length=64,
    batch_size=32
):
    all_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i:i + batch_size]

        encoded = tokenizer(
            batch_sentences,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            model_output = model(**encoded)

        embeddings = mean_pooling(
            model_output,
            encoded["attention_mask"]
        )

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu()
        )

    return torch.cat(all_embeddings, dim=0).numpy()

In [10]:
sentence1_embeddings = get_sentence_embedding(
    list(dataset["validation"]["sentence1"]),
    max_length=64
)

sentence2_embeddings = get_sentence_embedding(
    list(dataset["validation"]["sentence2"]),
    max_length=64
)

print("Sentence 1:", sentence1_embeddings.shape)
print("Sentence 2:", sentence2_embeddings.shape)

Sentence 1: (519, 768)
Sentence 2: (519, 768)


In [11]:
sts_predictions = np.sum(
    sentence1_embeddings * sentence2_embeddings,
    axis=1
)

print("Prediction shape:", sts_predictions.shape)
print("First 10 predictions:")
print(sts_predictions[:10])

Prediction shape: (519,)
First 10 predictions:
[0.94822025 0.92022544 0.90165734 0.89067566 0.87234604 0.8895432
 0.91061723 0.8838362  0.80825543 0.83347374]


In [12]:
gold_scores = np.array(
    [
        x["real-label"]
        for x in dataset["validation"]["labels"]
    ],
    dtype=np.float32
)

sts_predictions = np.asarray(
    sts_predictions,
    dtype=np.float32
)

print("Gold dtype:", gold_scores.dtype)
print("Prediction dtype:", sts_predictions.dtype)

Gold dtype: float32
Prediction dtype: float32


In [13]:
from scipy.stats import pearsonr, spearmanr

pearson = pearsonr(
    gold_scores,
    sts_predictions
).statistic

spearman = spearmanr(
    gold_scores,
    sts_predictions
).statistic

print(f"Pearson : {pearson:.4f}")
print(f"Spearman: {spearman:.4f}")

Pearson : 0.3475
Spearman: 0.3558


# Day 1 Experiment Log

Established a pretrained sentence-embedding baseline for later comparison.


# Day 2 - KLUE-STS Pair Dataset & Similarity Regression

Fine-tune a shared encoder with similarity regression.


In [14]:
# STS similarity score 추출

train_scores = np.array(
    [
        x["real-label"]
        for x in dataset["train"]["labels"]
    ],
    dtype=np.float32
)

print("Number of scores:", len(train_scores))
print("Mean:", train_scores.mean())
print("Std :", train_scores.std())
print("Min :", train_scores.min())
print("Max :", train_scores.max())

Number of scores: 11668
Mean: 2.2165613
Std : 1.7498462
Min : 0.0
Max : 5.0


In [15]:
# Similarity score 분포 확인
percentiles = [0, 10, 25, 50, 75, 90, 95, 99, 100]

for p in percentiles:
    print(f"{p}%: {np.percentile(train_scores, p):.2f}")

0%: 0.00
10%: 0.00
25%: 0.20
50%: 2.67
75%: 3.86
90%: 4.29
95%: 4.50
99%: 4.80
100%: 5.00


In [16]:
from datasets import Dataset, DatasetDict

def prepare_sts_dataset(dataset):
    def convert_split(split):
        return Dataset.from_dict({
            "sentence1": split["sentence1"],
            "sentence2": split["sentence2"],
            "score": [
                float(x["real-label"])
                for x in split["labels"]
            ]
        })

    return DatasetDict({
        "train": convert_split(dataset["train"]),
        "validation": convert_split(dataset["validation"])
    })


sts_dataset = prepare_sts_dataset(dataset)



In [17]:
MAX_LENGTH = 64
def tokenize_sts_pair(example):
    encoded_1 = tokenizer(
        example["sentence1"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded_2 = tokenizer(
        example["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    return {
        "input_ids_1": encoded_1["input_ids"],
        "attention_mask_1": encoded_1["attention_mask"],
        "input_ids_2": encoded_2["input_ids"],
        "attention_mask_2": encoded_2["attention_mask"],
        "labels": example["score"],
    }


tokenized_sts_dataset = sts_dataset.map(
    tokenize_sts_pair,
    remove_columns=sts_dataset["train"].column_names
)

Map:   0%|          | 0/11668 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

In [18]:
from torch.utils.data import DataLoader

train_dataset = tokenized_sts_dataset["train"]
validation_dataset = tokenized_sts_dataset["validation"]

BATCH_SIZE = 32

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids_1",
        "attention_mask_1",
        "input_ids_2",
        "attention_mask_2",
        "labels"
    ]
)

validation_dataset.set_format(
    type="torch",
    columns=[
        "input_ids_1",
        "attention_mask_1",
        "input_ids_2",
        "attention_mask_2",
        "labels"
    ]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [20]:
import torch.nn as nn
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")

class RobertaSTSModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)

    def encode(self, input_ids, attention_mask):
        model_output = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        embedding = mean_pooling(
            model_output,
            attention_mask
        )

        return embedding

    def forward(
        self,
        input_ids_1,
        attention_mask_1,
        input_ids_2,
        attention_mask_2
    ):
        embedding_1 = self.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = self.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        )

        return embedding_1, embedding_2, similarity

In [21]:
model = RobertaSTSModel(model_name)
model = model.to(device)
print(model.roberta.config.hidden_size)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


768


In [22]:
from torch.optim import AdamW

LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Optimizer:", optimizer)

Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0.01
)


In [23]:
def calculate_loss(similarity, labels):
    predicted_score = (similarity + 1) / 2
    normalized_labels = labels / 5.0

    loss_fn = nn.MSELoss()

    loss = loss_fn(
        predicted_score,
        normalized_labels
    )

    return loss

In [24]:
NUM_EPOCHS = 1

model.train()

total_train_loss = 0.0

for batch_idx, batch in enumerate(train_loader):

    input_ids_1 = batch["input_ids_1"].to(device)
    attention_mask_1 = batch["attention_mask_1"].to(device)

    input_ids_2 = batch["input_ids_2"].to(device)
    attention_mask_2 = batch["attention_mask_2"].to(device)

    labels = batch["labels"].to(device)

    # 이전 batch의 gradient 초기화
    optimizer.zero_grad()

    embedding_1, embedding_2, similarity = model(
        input_ids_1=input_ids_1,
        attention_mask_1=attention_mask_1,
        input_ids_2=input_ids_2,
        attention_mask_2=attention_mask_2
    )

    loss = calculate_loss(
        similarity,
        labels
    )

    loss.backward()

    optimizer.step()

    total_train_loss += loss.item()

    # 진행 상황 출력
    if (batch_idx + 1) % 100 == 0:
        print(
            f"Batch [{batch_idx + 1}/{len(train_loader)}] "
            f"Loss: {loss.item():.4f}"
        )

average_train_loss = total_train_loss / len(train_loader)

print("Training complete")
print(f"Average Train Loss: {average_train_loss:.4f}")

Batch [100/365] Loss: 0.1092
Batch [200/365] Loss: 0.1150
Batch [300/365] Loss: 0.0913
Training complete
Average Train Loss: 0.1127


In [25]:
model.eval()

total_val_loss = 0.0

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in validation_loader:

        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)

        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)

        labels = batch["labels"].to(device)

        embedding_1, embedding_2, similarity = model(
            input_ids_1=input_ids_1,
            attention_mask_1=attention_mask_1,
            input_ids_2=input_ids_2,
            attention_mask_2=attention_mask_2
        )

        loss = calculate_loss(
            similarity,
            labels
        )

        total_val_loss += loss.item()

        # 0~1 -> 실제 STS score 0~5
        predicted_score = ((similarity + 1) / 2) * 5

        all_predictions.extend(
            predicted_score.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

average_val_loss = total_val_loss / len(validation_loader)

print("Validation complete")
print(f"Average Validation Loss: {average_val_loss:.4f}")

Validation complete
Average Validation Loss: 0.0832


In [26]:
pearson = pearsonr(
    all_labels,
    all_predictions
)[0]

spearman = spearmanr(
    all_labels,
    all_predictions
)[0]

print(f"Pearson Correlation:  {pearson:.4f}")
print(f"Spearman Correlation: {spearman:.4f}")

Pearson Correlation:  0.7096
Spearman Correlation: 0.7136


# Day 2 Experiment Log

Fine-tuned the encoder with supervised similarity regression.


# Day 3 - Margin-based Contrastive Fine-tuning

Train contrastive sentence embeddings with positive and negative pairs.


In [27]:
model = RobertaSTSModel(model_name)
model = model.to(device)

print("Hidden Size:", model.roberta.config.hidden_size)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden Size: 768


In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0.01
)


In [29]:
def contrastive_loss(
    embedding_1,
    embedding_2,
    temperature=0.05
):
    embedding_1 = F.normalize(
        embedding_1,
        p=2,
        dim=1
    )

    embedding_2 = F.normalize(
        embedding_2,
        p=2,
        dim=1
    )

    similarity_matrix = torch.matmul(
        embedding_1,
        embedding_2.T
    )

    similarity_matrix = similarity_matrix / temperature

    labels = torch.arange(
        embedding_1.size(0),
        device=embedding_1.device
    )

    loss_1 = F.cross_entropy(
        similarity_matrix,
        labels
    )

    loss_2 = F.cross_entropy(
        similarity_matrix.T,
        labels
    )

    loss = (loss_1 + loss_2) / 2

    return loss


model.train()

batch = next(iter(train_loader))

input_ids_1 = batch["input_ids_1"].to(device)
attention_mask_1 = batch["attention_mask_1"].to(device)

input_ids_2 = batch["input_ids_2"].to(device)
attention_mask_2 = batch["attention_mask_2"].to(device)

embedding_1 = model.encode(
    input_ids_1,
    attention_mask_1
)

embedding_2 = model.encode(
    input_ids_2,
    attention_mask_2
)

print("Embedding 1:", embedding_1.shape)
print("Embedding 2:", embedding_2.shape)

loss = contrastive_loss(
    embedding_1,
    embedding_2
)

print("Initial Contrastive Loss:", loss.item())

Embedding 1: torch.Size([32, 768])
Embedding 2: torch.Size([32, 768])
Initial Contrastive Loss: 1.575196623802185


In [30]:
model.train()

epochs = 1

for epoch in range(epochs):

    total_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):

        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)

        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)

        optimizer.zero_grad()

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        loss = contrastive_loss(
            embedding_1,
            embedding_2
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] "
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Average Train Loss: {average_loss:.4f}"
    )

Epoch [1/1] Batch [100/365] Loss: 0.9418
Epoch [1/1] Batch [200/365] Loss: 0.6722
Epoch [1/1] Batch [300/365] Loss: 0.7307
Epoch [1/1] Average Train Loss: 0.6506


In [31]:
model.eval()

all_predictions = []
all_labels = []

total_loss = 0.0
num_batches = 0

with torch.no_grad():

    for batch in validation_loader:

        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)

        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)

        labels = batch["labels"].to(device)

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        )

        predictions = (similarity + 1) / 2
        normalized_labels = labels / 5.0

        loss = F.mse_loss(
            predictions,
            normalized_labels
        )

        total_loss += loss.item()
        num_batches += 1

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            normalized_labels.cpu().numpy()
        )


average_val_loss = total_loss / num_batches

pearson_score = pearsonr(
    all_predictions,
    all_labels
)[0]

spearman_score = spearmanr(
    all_predictions,
    all_labels
)[0]

print(f"Validation Loss: {average_val_loss:.4f}")
print(f"Pearson Correlation: {pearson_score:.4f}")
print(f"Spearman Correlation: {spearman_score:.4f}")

Validation Loss: 0.2239
Pearson Correlation: 0.1746
Spearman Correlation: 0.1945


### Positive / Negative Pair Construction

KLUE-STS provides a continuous similarity score from 0 to 5.

To apply margin-based contrastive learning, the continuous similarity score was converted into a binary positive/negative relationship.

A similarity score of `3.0` was selected as the experimental threshold:

- Score >= 3.0 -> Positive pair
- Score < 3.0 -> Negative pair

The threshold of `3.0` was an experimental design choice for converting the continuous STS scores into binary pair relationships. It was not treated as an official KLUE-STS classification threshold.

The contrastive loss margin was set to `0.5`.

In [32]:
model = RobertaSTSModel(model_name).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

positive_threshold = 3.0
margin = 0.5

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
def margin_contrastive_loss(
    embedding_1,
    embedding_2,
    labels,
    margin=0.5
):
    embedding_1 = F.normalize(
        embedding_1,
        p=2,
        dim=1
    )

    embedding_2 = F.normalize(
        embedding_2,
        p=2,
        dim=1
    )

    distance = 1 - F.cosine_similarity(
        embedding_1,
        embedding_2,
        dim=1
    )

    positive_mask = labels >= 3.0
    negative_mask = labels < 3.0

    positive_loss = distance[positive_mask].pow(2)

    negative_loss = F.relu(
        margin - distance[negative_mask]
    ).pow(2)

    losses = []

    if positive_loss.numel() > 0:
        losses.append(positive_loss.mean())

    if negative_loss.numel() > 0:
        losses.append(negative_loss.mean())

    if len(losses) == 0:
        return torch.tensor(
            0.0,
            device=embedding_1.device,
            requires_grad=True
        )

    return torch.stack(losses).mean()

In [34]:
model.train()

epochs = 1

for epoch in range(epochs):

    total_loss = 0.0
    valid_batches = 0

    for batch_idx, batch in enumerate(train_loader):

        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)

        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        loss = margin_contrastive_loss(
            embedding_1,
            embedding_2,
            labels,
            margin=0.5
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        valid_batches += 1

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch + 1}/{epochs}] "
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / valid_batches

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Average Train Loss: {average_loss:.4f}"
    )

Epoch [1/1] Batch [100/365] Loss: 0.0171
Epoch [1/1] Batch [200/365] Loss: 0.0164
Epoch [1/1] Batch [300/365] Loss: 0.0066
Epoch [1/1] Average Train Loss: 0.0144


In [35]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in validation_loader:

        input_ids_1 = batch["input_ids_1"].to(device)
        attention_mask_1 = batch["attention_mask_1"].to(device)

        input_ids_2 = batch["input_ids_2"].to(device)
        attention_mask_2 = batch["attention_mask_2"].to(device)

        labels = batch["labels"].to(device)

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        )

        predictions = (similarity + 1) / 2 * 5

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


pearson_score = pearsonr(
    all_predictions,
    all_labels
)[0]

spearman_score = spearmanr(
    all_predictions,
    all_labels
)[0]

print(f"Pearson Correlation: {pearson_score:.4f}")
print(f"Spearman Correlation: {spearman_score:.4f}")

Pearson Correlation: 0.8333
Spearman Correlation: 0.8531


In [36]:
import copy

day3_model_state = copy.deepcopy(model.state_dict())
print("Day 3 model snapshot saved for multi-seed final training.")


Day 3 model snapshot saved for multi-seed final training.


# Day 3 Experiment Log

Shaped the embedding space with contrastive learning before hard-negative mining.


# Day 4 - Hard Negative Construction & Analysis

Mine hard negatives and select a robust retriever configuration.


In [37]:
HARD_NEGATIVE_THRESHOLD = 3.0

hard_negative_candidates = sts_dataset["train"].filter(
    lambda example: example["score"] < HARD_NEGATIVE_THRESHOLD
)

print("Total training pairs:", len(sts_dataset["train"]))
print("Negative candidates:", len(hard_negative_candidates))

print("First 5 negative candidates:")

for i in range(min(5, len(hard_negative_candidates))):
    example = hard_negative_candidates[i]

    print(f"[{i}]")
    print("Sentence 1:", example["sentence1"])
    print("Sentence 2:", example["sentence2"])
    print("Score:", example["score"])

Filter:   0%|          | 0/11668 [00:00<?, ? examples/s]

Total training pairs: 11668
Negative candidates: 6066
First 5 negative candidates:
[0]
Sentence 1: 위반행위 조사 등을 거부·방해·기피한 자는 500만원 이하 과태료 부과 대상이다.
Sentence 2: 시민들 스스로 자발적인 예방 노력을 한 것은 아산 뿐만이 아니었다.
Score: 0.0
[1]
Sentence 1: 회사가 보낸 메일은 이 지메일이 아니라 다른 지메일 계정으로 전달해줘.
Sentence 2: 사람들이 주로 네이버 메일을 쓰는 이유를 알려줘
Score: 0.3333333333333333
[2]
Sentence 1: 긴급 고용안정지원금은 지역고용대응 등 특별지원금, 지자체별 소상공인 지원사업, 취업성공패키지, 청년구직활동지원금, 긴급복지지원제도 지원금과는 중복 수급이 불가능하다.
Sentence 2: 고용보험이 1차 고용안전망이라면, 국민취업지원제도는 2차 고용안전망입니다.
Score: 0.5714285714285714
[3]
Sentence 1: 정부가 새로운 일자리를 직접 창출하는 노력도 배가하겠습니다.
Sentence 2: 세계에서 우리만큼 오랜 역사와 문화를 공유하는 가까운 이웃이 없습니다.
Score: 0.0
[4]
Sentence 1: 사례집은 국립환경과학원 누리집(ecolibrary.me.go.kr)에서 12일부터 볼 수 있다.
Sentence 2: 주말을 제외한 평일 오후 12시 30분부터 문예회관 공식 페이스북과 유튜브에서는 지역 예술인들이 중심이 된 서양음악, 국악, 댄스 등의 공연을 실시간으로 감상할 수 있다.
Score: 0.0


In [38]:
from tqdm.auto import tqdm

model.eval()

hard_negative_scores = []

with torch.no_grad():

    for example in tqdm(
        hard_negative_candidates,
        desc="Calculating cosine similarities"
    ):
        sentence1 = example["sentence1"]
        sentence2 = example["sentence2"]

        encoded1 = tokenizer(
            sentence1,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        encoded2 = tokenizer(
            sentence2,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        # GPU로 이동
        input_ids_1 = encoded1["input_ids"].to(device)
        attention_mask_1 = encoded1["attention_mask"].to(device)

        input_ids_2 = encoded2["input_ids"].to(device)
        attention_mask_2 = encoded2["attention_mask"].to(device)

        # Day 3 모델로 embedding 생성
        embedding1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding1,
            embedding2,
            dim=1
        ).item()

        hard_negative_scores.append({
            "sentence1": sentence1,
            "sentence2": sentence2,
            "sts_score": example["score"],
            "cosine_similarity": similarity
        })

print(
    "Number of evaluated negative pairs:",
    len(hard_negative_scores)
)

Calculating cosine similarities:   0%|          | 0/6066 [00:00<?, ?it/s]

Number of evaluated negative pairs: 6066


In [39]:
HARD_SIMILARITY_THRESHOLD = 0.90

hard_negative_candidates_filtered = [
    x
    for x in hard_negative_scores
    if x["cosine_similarity"] >= HARD_SIMILARITY_THRESHOLD
]

print(
    "Total negative candidates:",
    len(hard_negative_scores)
)

print(
    "High-similarity candidates:",
    len(hard_negative_candidates_filtered)
)

Total negative candidates: 6066
High-similarity candidates: 78


In [40]:
hard_negative_review_candidates = [
    x
    for x in hard_negative_scores
    if x["cosine_similarity"] >= 0.90
    and 2.0 <= x["sts_score"] < 3.0
]

# cosine similarity가 높은 순서로 정렬
hard_negative_review_candidates = sorted(
    hard_negative_review_candidates,
    key=lambda x: x["cosine_similarity"],
    reverse=True
)

print(
    "Hard Negative review candidates:",
    len(hard_negative_review_candidates)
)

Hard Negative review candidates: 76


In [41]:
hard_negative_dataset = Dataset.from_list(
    [
        {
            "sentence1": x["sentence1"],
            "sentence2": x["sentence2"],
            "sts_score": float(x["sts_score"]),
            "cosine_similarity": float(x["cosine_similarity"]),
            "label": 0
        }
        for x in hard_negative_review_candidates
    ]
)


print("First 5 candidates:")

for i in range(min(5, len(hard_negative_dataset))):
    print(f"[{i}]")
    print("Sentence 1:", hard_negative_dataset[i]["sentence1"])
    print("Sentence 2:", hard_negative_dataset[i]["sentence2"])
    print("STS score:", hard_negative_dataset[i]["sts_score"])
    print(
        "Cosine similarity:",
        hard_negative_dataset[i]["cosine_similarity"]
    )
    print("Label:", hard_negative_dataset[i]["label"])

First 5 candidates:
[0]
Sentence 1: 겨울에 사람 만날 만한 곳이 어디인지 알려줄 수 있을까?
Sentence 2: 어느 장소가 겨울에 만나기 좋습니까?
STS score: 2.571428571428572
Cosine similarity: 0.9699188470840454
Label: 0
[1]
Sentence 1: 세탁기, 식기세척기, 냉장고 이용 가능했어요
Sentence 2: 세탁기, 식기세척기, 냉장고요.
STS score: 2.428571428571428
Cosine similarity: 0.9697148203849792
Label: 0
[2]
Sentence 1: 내가 쓴 메일들 중에 학교로 보내고 사월에 보낸 것들의 개수를 알려줘
Sentence 2: 제 보낸 편지함에 사월에 학교로 보낸 메일이 몇통이나 있나요?
STS score: 2.5
Cosine similarity: 0.9692856669425964
Label: 0
[3]
Sentence 1: 저는 이 곳에 머물었던 기억때문에 시드니를 다시 올 것 같을 정도로 행복했습니다.
Sentence 2: 저는 시드니에 다시 오게 된 것이 너무 행복했습니다. 왜냐하면 이곳에 머물렀던 제 기억들 때문입니다.
STS score: 2.833333333333333
Cosine similarity: 0.9679800271987915
Label: 0
[4]
Sentence 1: 한국분 중에 로마로 방문 계획이 있으시다면, 강추 드립니다.
Sentence 2: 로마를 방문할 계획이 있으신 분은 꼭 추천해 드립니다.
STS score: 2.833333333333333
Cosine similarity: 0.9675059914588928
Label: 0


In [42]:
POSITIVE_THRESHOLD = 3.0

# 기존 STS train에서 Positive 추출
positive_examples = sts_dataset["train"].filter(
    lambda example: example["score"] >= POSITIVE_THRESHOLD
)

print("Original positive pairs:", len(positive_examples))
print("Hard negative pairs:", len(hard_negative_dataset))

Filter:   0%|          | 0/11668 [00:00<?, ? examples/s]

Original positive pairs: 5602
Hard negative pairs: 76


In [43]:
positive_training_data = [
    {
        "sentence1": example["sentence1"],
        "sentence2": example["sentence2"],
        "label": 1,
        "sts_score": float(example["score"]),
        "cosine_similarity": None
    }
    for example in positive_examples
]

# Hard Negative 추가
hard_negative_training_data = [
    {
        "sentence1": example["sentence1"],
        "sentence2": example["sentence2"],
        "label": 0,
        "sts_score": float(example["sts_score"]),
        "cosine_similarity": float(
            example["cosine_similarity"]
        )
    }
    for example in hard_negative_dataset
]

# 합치기
day11_training_data = (
    positive_training_data +
    hard_negative_training_data
)

print("Total Day 4 training pairs:", len(day11_training_data))
print(
    "Positive:",
    sum(x["label"] == 1 for x in day11_training_data)
)
print(
    "Hard Negative:",
    sum(x["label"] == 0 for x in day11_training_data)
)

Total Day 4 training pairs: 5678
Positive: 5602
Hard Negative: 76


In [44]:
import random

random.seed(SEED)

positive_sample = random.sample(
    positive_training_data,
    len(hard_negative_training_data)
)

day11_balanced_data = (
    positive_sample +
    hard_negative_training_data
)

# 순서 섞기
random.shuffle(day11_balanced_data)

print("Day 4 balanced dataset")
print("Total:", len(day11_balanced_data))
print(
    "Positive:",
    sum(x["label"] == 1 for x in day11_balanced_data)
)
print(
    "Hard Negative:",
    sum(x["label"] == 0 for x in day11_balanced_data)
)

Day 4 balanced dataset
Total: 152
Positive: 76
Hard Negative: 76


In [45]:
from torch.utils.data import Dataset, DataLoader

class Day11PairDataset(Dataset):

    def __init__(self, data, tokenizer, max_length=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        example = self.data[idx]

        encoded1 = self.tokenizer(
            example["sentence1"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        encoded2 = self.tokenizer(
            example["sentence2"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids_1": encoded1["input_ids"].squeeze(0),
            "attention_mask_1": encoded1["attention_mask"].squeeze(0),

            "input_ids_2": encoded2["input_ids"].squeeze(0),
            "attention_mask_2": encoded2["attention_mask"].squeeze(0),

            "label": torch.tensor(
                example["label"],
                dtype=torch.float
            )
        }


day11_dataset = Day11PairDataset(
    day11_balanced_data,
    tokenizer,
    max_length=64
)

day11_loader = DataLoader(
    day11_dataset,
    batch_size=16,
    shuffle=True
)

print("Dataset size:", len(day11_dataset))
print("Number of batches:", len(day11_loader))

Dataset size: 152
Number of batches: 10


In [46]:
for example in day11_balanced_data:

    if example["label"] == 1:
        example["label"] = 3.0
    else:
        example["label"] = 0.0


print("Label distribution:")

print(
    "Positive (3.0):",
    sum(x["label"] == 3.0 for x in day11_balanced_data)
)

print(
    "Hard Negative (0.0):",
    sum(x["label"] == 0.0 for x in day11_balanced_data)
)

Label distribution:
Positive (3.0): 76
Hard Negative (0.0): 76


In [47]:
model.train()

total_loss = 0.0
num_batches = 0

for batch in tqdm(day11_loader, desc="Day 4 Fine-tuning"):

    # GPU 이동
    input_ids_1 = batch["input_ids_1"].to(device)
    attention_mask_1 = batch["attention_mask_1"].to(device)

    input_ids_2 = batch["input_ids_2"].to(device)
    attention_mask_2 = batch["attention_mask_2"].to(device)

    labels = batch["label"].to(device)

    # 기존 optimizer 사용
    optimizer.zero_grad()

    embedding_1 = model.encode(
        input_ids_1,
        attention_mask_1
    )

    embedding_2 = model.encode(
        input_ids_2,
        attention_mask_2
    )

    # 기존 Day 3 contrastive loss
    loss = margin_contrastive_loss(
        embedding_1,
        embedding_2,
        labels
    )

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
    num_batches += 1

average_loss = total_loss / num_batches

print("Day 4 Fine-tuning completed")
print(f"Average loss: {average_loss:.6f}")
print(f"Epochs: 1")
print(f"Training pairs: {len(day11_dataset)}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")

Day 4 Fine-tuning:   0%|          | 0/10 [00:00<?, ?it/s]

Day 4 Fine-tuning completed
Average loss: 0.069971
Epochs: 1
Training pairs: 152
Learning rate: 2e-05


In [48]:
model.eval()

hard_negative_after = []

with torch.no_grad():

    for example in hard_negative_dataset:

        encoded1 = tokenizer(
            example["sentence1"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        encoded2 = tokenizer(
            example["sentence2"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        input_ids_1 = encoded1["input_ids"].to(device)
        attention_mask_1 = encoded1["attention_mask"].to(device)

        input_ids_2 = encoded2["input_ids"].to(device)
        attention_mask_2 = encoded2["attention_mask"].to(device)

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        ).item()

        hard_negative_after.append(similarity)

print("Number of evaluated pairs:", len(hard_negative_after))

Number of evaluated pairs: 76


In [49]:
before_scores = [
    example["cosine_similarity"]
    for example in hard_negative_dataset
]

after_scores = hard_negative_after

changes = [
    after - before
    for before, after in zip(before_scores, after_scores)
]

print("Hard Negative similarity: Before vs After")

print(f"Before mean : {sum(before_scores) / len(before_scores):.4f}")
print(f"After mean  : {sum(after_scores) / len(after_scores):.4f}")
print(f"Mean change : {sum(changes) / len(changes):.4f}")

print(f"Before max  : {max(before_scores):.4f}")
print(f"After max   : {max(after_scores):.4f}")

print("First 10 examples:")

for i in range(min(10, len(before_scores))):

    print(
        f"[{i+1:02d}] "
        f"{before_scores[i]:.4f} -> "
        f"{after_scores[i]:.4f} "
        f"(Δ {changes[i]:+.4f})"
    )

Hard Negative similarity: Before vs After
Before mean : 0.9298
After mean  : 0.7267
Mean change : -0.2031
Before max  : 0.9699
After max   : 0.9162
First 10 examples:
[01] 0.9699 -> 0.7287 (Δ -0.2412)
[02] 0.9697 -> 0.7833 (Δ -0.1864)
[03] 0.9693 -> 0.8677 (Δ -0.1016)
[04] 0.9680 -> 0.8854 (Δ -0.0825)
[05] 0.9675 -> 0.8502 (Δ -0.1173)
[06] 0.9645 -> 0.7769 (Δ -0.1876)
[07] 0.9643 -> 0.8999 (Δ -0.0644)
[08] 0.9638 -> 0.8206 (Δ -0.1432)
[09] 0.9632 -> 0.8666 (Δ -0.0966)
[10] 0.9583 -> 0.7341 (Δ -0.2242)


In [50]:
model.eval()

positive_after = []

with torch.no_grad():

    for example in positive_sample:

        encoded1 = tokenizer(
            example["sentence1"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        encoded2 = tokenizer(
            example["sentence2"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        input_ids_1 = encoded1["input_ids"].to(device)
        attention_mask_1 = encoded1["attention_mask"].to(device)

        input_ids_2 = encoded2["input_ids"].to(device)
        attention_mask_2 = encoded2["attention_mask"].to(device)

        embedding_1 = model.encode(
            input_ids_1,
            attention_mask_1
        )

        embedding_2 = model.encode(
            input_ids_2,
            attention_mask_2
        )

        similarity = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        ).item()

        positive_after.append(similarity)

print("Positive similarity after Day 4 fine-tuning")

print(
    f"Mean : "
    f"{sum(positive_after) / len(positive_after):.4f}"
)

print(
    f"Min  : "
    f"{min(positive_after):.4f}"
)

print(
    f"Max  : "
    f"{max(positive_after):.4f}"
)

Positive similarity after Day 4 fine-tuning
Mean : 0.9175
Min  : 0.7680
Max  : 0.9972


In [51]:
model.eval()

predicted_scores = []
true_scores = []

with torch.no_grad():

    for example in tqdm(
        sts_dataset["validation"],
        desc="STS Evaluation"
    ):

        encoded1 = tokenizer(
            example["sentence1"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        encoded2 = tokenizer(
            example["sentence2"],
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        embedding_1 = model.encode(
            encoded1["input_ids"].to(device),
            encoded1["attention_mask"].to(device)
        )

        embedding_2 = model.encode(
            encoded2["input_ids"].to(device),
            encoded2["attention_mask"].to(device)
        )

        cosine = F.cosine_similarity(
            embedding_1,
            embedding_2,
            dim=1
        ).item()

        predicted_scores.append(cosine)
        true_scores.append(float(example["score"]))

print("Day 4 STS Evaluation")

print("Number of pairs:", len(predicted_scores))

print("Prediction range:")
print(f"Min: {min(predicted_scores):.4f}")
print(f"Max: {max(predicted_scores):.4f}")

print("True STS score range:")
print(f"Min: {min(true_scores):.4f}")
print(f"Max: {max(true_scores):.4f}")

STS Evaluation:   0%|          | 0/519 [00:00<?, ?it/s]

Day 4 STS Evaluation
Number of pairs: 519
Prediction range:
Min: 0.0139
Max: 0.9941
True STS score range:
Min: 0.0000
Max: 5.0000


In [52]:
pearson_result = pearsonr(
    predicted_scores,
    true_scores
)

spearman_result = spearmanr(
    predicted_scores,
    true_scores
)

print("Day 4 STS Correlation Evaluation")

print(f"Pearson correlation  : {pearson_result.statistic:.4f}")
print(f"Spearman correlation : {spearman_result.statistic:.4f}")

Day 4 STS Correlation Evaluation
Pearson correlation  : 0.8370
Spearman correlation : 0.8571


In [53]:
import copy
from torch.utils.data import DataLoader

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

def evaluate_encoder(encoder_model):
    encoder_model.eval()
    predictions = []
    labels = []
    with torch.no_grad():
        for batch in validation_loader:
            embedding_1 = encoder_model.encode(
                batch["input_ids_1"].to(device),
                batch["attention_mask_1"].to(device)
            )
            embedding_2 = encoder_model.encode(
                batch["input_ids_2"].to(device),
                batch["attention_mask_2"].to(device)
            )
            similarity = F.cosine_similarity(
                embedding_1, embedding_2, dim=1
            )
            predictions.extend(((similarity + 1) / 2 * 5).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    return {
        "pearson": float(pearsonr(labels, predictions)[0]),
        "spearman": float(spearmanr(labels, predictions)[0])
    }

multi_seed_records = []
best_seed = None
best_seed_score = float("-inf")
best_seed_state = None

for seed in SEEDS:
    seed_everything(seed)
    seed_model = RobertaSTSModel(model_name).to(device)
    seed_model.load_state_dict(day3_model_state)
    seed_optimizer = torch.optim.AdamW(
        seed_model.parameters(), lr=2e-5, weight_decay=0.01
    )
    seed_loader = DataLoader(
        day11_dataset, batch_size=32, shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )
    seed_model.train()
    total_loss = 0.0
    for batch in seed_loader:
        seed_optimizer.zero_grad()
        embedding_1 = seed_model.encode(
            batch["input_ids_1"].to(device),
            batch["attention_mask_1"].to(device)
        )
        embedding_2 = seed_model.encode(
            batch["input_ids_2"].to(device),
            batch["attention_mask_2"].to(device)
        )
        loss = margin_contrastive_loss(
            embedding_1, embedding_2, batch["label"].to(device)
        )
        loss.backward()
        seed_optimizer.step()
        total_loss += loss.item()
    seed_metrics = evaluate_encoder(seed_model)
    seed_metrics.update({
        "seed": seed,
        "train_loss": total_loss / len(seed_loader),
        "training_pairs": len(day11_dataset)
    })
    multi_seed_records.append(seed_metrics)
    selection_score = (seed_metrics["pearson"] + seed_metrics["spearman"]) / 2
    if selection_score > best_seed_score:
        best_seed = seed
        best_seed_score = selection_score
        best_seed_state = copy.deepcopy(seed_model.state_dict())
    print(f"Seed {seed}: {seed_metrics}")

model.load_state_dict(best_seed_state)
model.eval()
multi_seed_results_df = pd.DataFrame(multi_seed_records).sort_values(
    ["pearson", "spearman"], ascending=False
).reset_index(drop=True)
print("Multi-seed final training results")
display(multi_seed_results_df)
print("Selected seed:", best_seed)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Seed 17: {'pearson': 0.8493326306343079, 'spearman': 0.8681563881354848, 'seed': 17, 'train_loss': 0.07252504080533981, 'training_pairs': 152}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Seed 378: {'pearson': 0.8518576622009277, 'spearman': 0.8702530780680904, 'seed': 378, 'train_loss': 0.07022102996706962, 'training_pairs': 152}
Multi-seed final training results


,pearson,spearman,seed,train_loss,training_pairs
0,0.851858,0.870253,378,0.070221,152
1,0.849470,0.867990,47,0.068856,152
2,0.849333,0.868156,17,0.072525,152


Selected seed: 378


# Day 4 Experiment Log

Selected the hard-negative retriever configuration from the requested multi-seed runs.


# Day 5 - FAISS-Based Top-K Retrieval with Fine-Tuned Sentence Embeddings

Build a FAISS index and evaluate Top-K semantic retrieval.


In [54]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.3 MB/s eta 0:00:00


In [55]:
import faiss

print("FAISS version:", faiss.__version__)

FAISS version: 1.15.0


In [56]:
import time

queries = list(
    sts_dataset["validation"]["sentence1"]
)

corpus = list(
    sts_dataset["validation"]["sentence2"]
)

gold_indices = np.arange(
    len(queries),
    dtype=np.int64
)

print("Day 5 Retrieval Dataset")
print("Number of queries:", len(queries))
print("Corpus size:", len(corpus))
print("First query:", queries[0])
print("Gold target:", corpus[gold_indices[0]])

Day 5 Retrieval Dataset
Number of queries: 519
Corpus size: 519
First query: 무엇보다도 호스트분들이 너무 친절하셨습니다.
Gold target: 무엇보다도, 호스트들은 매우 친절했습니다.


In [57]:
def encode_sentences_with_model(
    sentences,
    model,
    tokenizer,
    device,
    batch_size=32,
    max_length=64
):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sentences = sentences[i:i + batch_size]

            encoded = tokenizer(
                batch_sentences,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            embeddings = model.encode(
                encoded["input_ids"],
                encoded["attention_mask"]
            )

            embeddings = F.normalize(
                embeddings,
                p=2,
                dim=1
            )

            all_embeddings.append(
                embeddings.cpu().numpy()
            )

    return np.concatenate(
        all_embeddings,
        axis=0
    ).astype("float32")


query_embeddings = encode_sentences_with_model(
    queries,
    model,
    tokenizer,
    device
)

corpus_embeddings = encode_sentences_with_model(
    corpus,
    model,
    tokenizer,
    device
)

print("Query embedding shape:", query_embeddings.shape)
print("Corpus embedding shape:", corpus_embeddings.shape)

Query embedding shape: (519, 768)
Corpus embedding shape: (519, 768)


In [58]:
embedding_dimension = corpus_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)
index.add(corpus_embeddings)

print("FAISS index type:", type(index).__name__)
print("Embedding dimension:", index.d)
print("Number of indexed vectors:", index.ntotal)

FAISS index type: IndexFlatIP
Embedding dimension: 768
Number of indexed vectors: 519


In [59]:
top_k = 10

similarities, retrieved_indices = index.search(
    query_embeddings,
    top_k
)

print("Similarity matrix shape:", similarities.shape)
print("Retrieved index shape:", retrieved_indices.shape)
print("First query Top-10 indices:", retrieved_indices[0])
print("First query Top-10 similarities:", similarities[0])

Similarity matrix shape: (519, 10)
Retrieved index shape: (519, 10)
First query Top-10 indices: [  0  94  34 431 240 370 384 476 156 397]
First query Top-10 similarities: [0.9801465  0.93566895 0.9055537  0.87151194 0.8586727  0.77293736
 0.76858616 0.7595434  0.710308   0.7012402 ]


In [60]:
gold_indices = np.asarray(
    gold_indices,
    dtype=np.int64
)

def calculate_recall_at_k(
    retrieved_indices,
    gold_indices,
    k
):
    correct = 0

    for gold_index, retrieved in zip(
        gold_indices,
        retrieved_indices[:, :k]
    ):
        if gold_index in retrieved:
            correct += 1

    return correct / len(gold_indices)


recall_at_1 = calculate_recall_at_k(
    retrieved_indices,
    gold_indices,
    k=1
)

recall_at_5 = calculate_recall_at_k(
    retrieved_indices,
    gold_indices,
    k=5
)

recall_at_10 = calculate_recall_at_k(
    retrieved_indices,
    gold_indices,
    k=10
)

print("Retrieval Results")
print(f"Recall@1 : {recall_at_1:.4f}")
print(f"Recall@5 : {recall_at_5:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Retrieval Results
Recall@1 : 0.6146
Recall@5 : 0.7938
Recall@10: 0.8478


In [61]:
num_runs = 10

start_time = time.perf_counter()

for _ in range(num_runs):
    index.search(
        query_embeddings,
        top_k
    )

elapsed_time = time.perf_counter() - start_time

average_latency_ms = (
    elapsed_time / num_runs
) * 1000

latency_per_query_ms = (
    elapsed_time / (num_runs * len(queries))
) * 1000

print("FAISS Search Latency")
print(f"Average search latency: {average_latency_ms:.4f} ms")
print(f"Latency per query: {latency_per_query_ms:.6f} ms")

FAISS Search Latency
Average search latency: 3.2822 ms
Latency per query: 0.006324 ms


In [62]:
def get_rank(retrieved, gold_index):
    matches = np.where(retrieved == gold_index)[0]

    if len(matches) == 0:
        return None

    return int(matches[0]) + 1


failure_count = 0

for i in range(len(queries)):
    rank = get_rank(
        retrieved_indices[i],
        gold_indices[i]
    )

    if rank is None:
        print(f"Query {i}")
        print("Query:", queries[i])
        print("Gold target:", corpus[gold_indices[i]])
        print("Top-1 retrieved:", corpus[retrieved_indices[i][0]])
        print("Top-1 similarity:", similarities[i][0])
        print("-" * 70)

        failure_count += 1

        if failure_count >= 5:
            break

Query 5
Query: 방안에 필요한 시설이 모두 있어서 매우 편리합니다.
Gold target: 특히, 숙소 근처에 안전한 실내 주차장이 있어서 편리합니다.
Top-1 retrieved: 위치와 주변 편의시설 모두 좋았습니다.
Top-1 similarity: 0.699999
----------------------------------------------------------------------
Query 8
Query: 일본종교문화를 잘 이해하시는 분이 이용하시는 게 좋아요
Gold target: 그리고 슬리퍼를 하나 준비하시는 게 좋아요
Top-1 retrieved: 한국인가요? 일본인가요? 태풍의 경로는.
Top-1 similarity: 0.5089724
----------------------------------------------------------------------
Query 9
Query: 다음에 도쿄를 또 간다면 무조건 이 곳을 다시 찾을 겁니다.
Gold target: 독일을 다음에 또간다면 이숙소에 머물겁니다.
Top-1 retrieved: 일본에 위치한 동경은 날씨가 어떤가요?
Top-1 similarity: 0.5592191
----------------------------------------------------------------------
Query 13
Query: 2개의 테라스에서 감상하실수 있습니다.
Gold target: 사진과 같이 테라스에서는 멋있는 뷰를 감상하실 수 있습니다.
Top-1 retrieved: 세 팀이 2층에 한 개씩 두 층을 나눠 쓰는 것 같았어요.
Top-1 similarity: 0.6478298
----------------------------------------------------------------------
Query 18
Query: 전형적인 일본 가정의 모습을 볼 수 있습니다.
Gold target: 하지만 호스트의 적극적인 도움으로 불편함을 덜 수 있었습니다.
T

In [63]:
RETRIEVAL_SCORE_THRESHOLD = 3.0

validation_scores = np.array(
    sts_dataset["validation"]["score"],
    dtype=np.float32
)

positive_mask = (
    validation_scores >= RETRIEVAL_SCORE_THRESHOLD
)

positive_query_indices = np.where(
    positive_mask
)[0]

positive_queries = [
    queries[i]
    for i in positive_query_indices
]

positive_gold_indices = positive_query_indices.astype(
    np.int64
)

print("Filtered Retrieval Dataset")
print("Score threshold:", RETRIEVAL_SCORE_THRESHOLD)
print("Number of queries:", len(positive_queries))
print("Corpus size:", len(corpus))

Filtered Retrieval Dataset
Score threshold: 3.0
Number of queries: 220
Corpus size: 519


In [64]:
positive_query_embeddings = encode_sentences_with_model(
    positive_queries,
    model,
    tokenizer,
    device
)

positive_similarities, positive_retrieved_indices = (
    index.search(
        positive_query_embeddings,
        top_k
    )
)

print(
    "Positive query embedding shape:",
    positive_query_embeddings.shape
)
print(
    "Retrieved index shape:",
    positive_retrieved_indices.shape
)

Positive query embedding shape: (220, 768)
Retrieved index shape: (220, 10)


In [65]:
positive_recall_at_1 = calculate_recall_at_k(
    positive_retrieved_indices,
    positive_gold_indices,
    k=1
)

positive_recall_at_5 = calculate_recall_at_k(
    positive_retrieved_indices,
    positive_gold_indices,
    k=5
)

positive_recall_at_10 = calculate_recall_at_k(
    positive_retrieved_indices,
    positive_gold_indices,
    k=10
)

print("Filtered Retrieval Results")
print(f"Recall@1 : {positive_recall_at_1:.4f}")
print(f"Recall@5 : {positive_recall_at_5:.4f}")
print(f"Recall@10: {positive_recall_at_10:.4f}")

Filtered Retrieval Results
Recall@1 : 0.9500
Recall@5 : 0.9955
Recall@10: 0.9955


In [66]:
filtered_num_runs = 10

start_time = time.perf_counter()

for _ in range(filtered_num_runs):
    index.search(
        positive_query_embeddings,
        top_k
    )

elapsed_time = time.perf_counter() - start_time

filtered_average_latency_ms = (
    elapsed_time / filtered_num_runs
) * 1000

filtered_latency_per_query_ms = (
    elapsed_time
    / (filtered_num_runs * len(positive_queries))
) * 1000

print("Filtered Retrieval Latency")
print(
    f"Average search latency: "
    f"{filtered_average_latency_ms:.4f} ms"
)
print(
    f"Latency per query: "
    f"{filtered_latency_per_query_ms:.6f} ms"
)

Filtered Retrieval Latency
Average search latency: 2.3647 ms
Latency per query: 0.010749 ms


In [67]:
gold_ranks = []

for retrieved, gold_index in zip(
    positive_retrieved_indices,
    positive_gold_indices
):
    rank = get_rank(
        retrieved,
        gold_index
    )

    if rank is not None:
        gold_ranks.append(rank)
    else:
        gold_ranks.append(None)

rank_distribution = {
    "Rank 1": 0,
    "Rank 2-5": 0,
    "Rank 6-10": 0,
    "Not in Top-10": 0
}

for rank in gold_ranks:
    if rank == 1:
        rank_distribution["Rank 1"] += 1
    elif rank is not None and rank <= 5:
        rank_distribution["Rank 2-5"] += 1
    elif rank is not None and rank <= 10:
        rank_distribution["Rank 6-10"] += 1
    else:
        rank_distribution["Not in Top-10"] += 1

print("Gold Target Rank Distribution")

for category, count in rank_distribution.items():
    print(f"{category}: {count}")

Gold Target Rank Distribution
Rank 1: 209
Rank 2-5: 10
Rank 6-10: 0
Not in Top-10: 1


In [68]:
failure_count = 0

for i, (retrieved, gold_index) in enumerate(
    zip(
        positive_retrieved_indices,
        positive_gold_indices
    )
):
    if retrieved[0] != gold_index:
        print(f"Query {i}")
        print("Query:", positive_queries[i])
        print("Gold target:", corpus[gold_index])
        print("Top-1 retrieved:", corpus[retrieved[0]])
        print(
            "Top-1 similarity:",
            positive_similarities[i][0]
        )
        print("-" * 70)

        failure_count += 1

        if failure_count >= 5:
            break

Query 10
Query: 집에 조명 조정하려면 어떻게 말해야해?
Gold target: 집안 조명 어떤 명령어로 조정하는지 알려줘
Top-1 retrieved: 집에 있는 조명 조정할 수 있는 명령어 알려줘
Top-1 similarity: 0.88653064
----------------------------------------------------------------------
Query 45
Query: 어머니 생일이 언제셔?
Gold target: 어머님이 태어나신 날짜가 어떻게 되나요?
Top-1 retrieved: 어머님 생신이 언제세요?
Top-1 similarity: 0.89932215
----------------------------------------------------------------------
Query 79
Query: 어머니께선 몇 년 몇 월 며칠에 태어나셨는지요?
Gold target: 어머님 생신이 언제세요?
Top-1 retrieved: 어머님이 태어나신 날짜가 어떻게 되나요?
Top-1 similarity: 0.87886167
----------------------------------------------------------------------
Query 106
Query: 특히 호스트가 엄청 친절하고 친절해요,
Gold target: 특히 사회자는 매우 친절하고 친절합니다.
Top-1 retrieved: 그리고 주인은 매우 친절하고 친절해요!
Top-1 similarity: 0.94708514
----------------------------------------------------------------------
Query 108
Query: 어떻게 집 습도를 측정하는거야?
Gold target: 집안 습도 측정방법 좀 알려줘
Top-1 retrieved: 집 습도는 어떤 식으로 잴 수 있어?
Top-1 similarity: 0.93004906
------------------------------

# Day 5 Experiment Log

The filtered retrieval benchmark reached Recall@1 0.9500 and Recall@5 0.9955 with the fine-tuned encoder.


# Day 6 - Pretrained vs Fine-Tuned Embedding Retrieval Comparison

Compare pretrained and fine-tuned retrieval performance.


In [69]:
fine_tuned_model = model

fine_tuned_model.eval()

print("Day 6 Comparison Setup")
print("Fine-tuned model:", type(fine_tuned_model).__name__)
print("Number of queries:", len(positive_queries))
print("Corpus size:", len(corpus))
print("Embedding dimension:", fine_tuned_model.roberta.config.hidden_size)
print("Maximum sequence length:", MAX_LENGTH)
print("Retrieval index:", "FAISS IndexFlatIP")
print("Top-K:", top_k)

Day 6 Comparison Setup
Fine-tuned model: RobertaSTSModel
Number of queries: 220
Corpus size: 519
Embedding dimension: 768
Maximum sequence length: 64
Retrieval index: FAISS IndexFlatIP
Top-K: 10


In [70]:
pretrained_model = RobertaSTSModel(
    "klue/roberta-base"
).to(device)

pretrained_model.eval()

print("Pretrained model loaded")
print("Model:", type(pretrained_model).__name__)
print(
    "Embedding dimension:",
    pretrained_model.roberta.config.hidden_size
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Pretrained model loaded
Model: RobertaSTSModel
Embedding dimension: 768


In [71]:
fine_tuned_query_embeddings = encode_sentences_with_model(
    positive_queries,
    fine_tuned_model,
    tokenizer,
    device
)

fine_tuned_corpus_embeddings = encode_sentences_with_model(
    corpus,
    fine_tuned_model,
    tokenizer,
    device
)

pretrained_query_embeddings = encode_sentences_with_model(
    positive_queries,
    pretrained_model,
    tokenizer,
    device
)

pretrained_corpus_embeddings = encode_sentences_with_model(
    corpus,
    pretrained_model,
    tokenizer,
    device
)

print("Embedding Generation Completed")
print(
    "Fine-tuned query embeddings:",
    fine_tuned_query_embeddings.shape
)
print(
    "Fine-tuned corpus embeddings:",
    fine_tuned_corpus_embeddings.shape
)
print(
    "Pretrained query embeddings:",
    pretrained_query_embeddings.shape
)
print(
    "Pretrained corpus embeddings:",
    pretrained_corpus_embeddings.shape
)

Embedding Generation Completed
Fine-tuned query embeddings: (220, 768)
Fine-tuned corpus embeddings: (519, 768)
Pretrained query embeddings: (220, 768)
Pretrained corpus embeddings: (519, 768)


In [72]:
def run_faiss_retrieval(
    query_embeddings,
    corpus_embeddings,
    top_k=10
):
    embedding_dimension = corpus_embeddings.shape[1]

    retrieval_index = faiss.IndexFlatIP(
        embedding_dimension
    )

    retrieval_index.add(
        corpus_embeddings
    )

    similarities, retrieved_indices = (
        retrieval_index.search(
            query_embeddings,
            top_k
        )
    )

    return (
        retrieval_index,
        similarities,
        retrieved_indices
    )


In [73]:
(
    fine_tuned_index,
    fine_tuned_similarities,
    fine_tuned_retrieved_indices
) = run_faiss_retrieval(
    fine_tuned_query_embeddings,
    fine_tuned_corpus_embeddings,
    top_k=10
)

(
    pretrained_index,
    pretrained_similarities,
    pretrained_retrieved_indices
) = run_faiss_retrieval(
    pretrained_query_embeddings,
    pretrained_corpus_embeddings,
    top_k=10
)

print("FAISS Retrieval Completed")
print(
    "Fine-tuned retrieved indices:",
    fine_tuned_retrieved_indices.shape
)
print(
    "Pretrained retrieved indices:",
    pretrained_retrieved_indices.shape
)

FAISS Retrieval Completed
Fine-tuned retrieved indices: (220, 10)
Pretrained retrieved indices: (220, 10)


In [74]:
def calculate_retrieval_metrics(
    retrieved_indices,
    gold_indices
):
    return {
        f"Recall@{k}": calculate_recall_at_k(
            retrieved_indices,
            gold_indices,
            k=k
        )
        for k in (1, 5, 10)
    }

fine_tuned_metrics = calculate_retrieval_metrics(
    fine_tuned_retrieved_indices,
    positive_gold_indices
)

pretrained_metrics = calculate_retrieval_metrics(
    pretrained_retrieved_indices,
    positive_gold_indices
)

print("Retrieval Metrics")
for metric in ("Recall@1", "Recall@5", "Recall@10"):
    print(
        f"{metric} | "
        f"Pretrained: {pretrained_metrics[metric]:.4f} | "
        f"Fine-tuned: {fine_tuned_metrics[metric]:.4f}"
    )


Retrieval Metrics
Recall@1 | Pretrained: 0.7318 | Fine-tuned: 0.9500
Recall@5 | Pretrained: 0.9045 | Fine-tuned: 0.9955
Recall@10 | Pretrained: 0.9545 | Fine-tuned: 0.9955


In [75]:
def measure_faiss_latency(
    index,
    query_embeddings,
    top_k=10,
    num_runs=10
):
    start_time = time.perf_counter()

    for _ in range(num_runs):
        index.search(
            query_embeddings,
            top_k
        )

    elapsed_time = (
        time.perf_counter() - start_time
    )

    average_latency_ms = (
        elapsed_time / num_runs
    ) * 1000

    latency_per_query_ms = (
        elapsed_time
        / (num_runs * len(query_embeddings))
    ) * 1000

    return (
        average_latency_ms,
        latency_per_query_ms
    )



pretrained_latency = measure_faiss_latency(
    pretrained_index,
    pretrained_query_embeddings,
    top_k=10
)

fine_tuned_latency = measure_faiss_latency(
    fine_tuned_index,
    fine_tuned_query_embeddings,
    top_k=10
)

print("Latency Comparison")
print(
    f"Pretrained average latency: "
    f"{pretrained_latency[0]:.4f} ms"
)
print(
    f"Fine-tuned average latency: "
    f"{fine_tuned_latency[0]:.4f} ms"
)
print(
    f"Pretrained latency per query: "
    f"{pretrained_latency[1]:.6f} ms"
)
print(
    f"Fine-tuned latency per query: "
    f"{fine_tuned_latency[1]:.6f} ms"
)

Latency Comparison
Pretrained average latency: 1.6439 ms
Fine-tuned average latency: 1.5518 ms
Pretrained latency per query: 0.007472 ms
Fine-tuned latency per query: 0.007054 ms


In [76]:
comparison_count = 0

for i in range(len(positive_queries)):
    gold_index = positive_gold_indices[i]

    pretrained_top1 = (
        pretrained_retrieved_indices[i][0]
    )

    fine_tuned_top1 = (
        fine_tuned_retrieved_indices[i][0]
    )

    pretrained_correct = (
        pretrained_top1 == gold_index
    )

    fine_tuned_correct = (
        fine_tuned_top1 == gold_index
    )

    if not pretrained_correct and fine_tuned_correct:
        print(f"Query {i}")
        print("Query:", positive_queries[i])
        print("Gold target:", corpus[gold_index])
        print(
            "Pretrained Top-1:",
            corpus[pretrained_top1]
        )
        print(
            "Pretrained similarity:",
            f"{pretrained_similarities[i][0]:.4f}"
        )
        print(
            "Fine-tuned Top-1:",
            corpus[fine_tuned_top1]
        )
        print(
            "Fine-tuned similarity:",
            f"{fine_tuned_similarities[i][0]:.4f}"
        )
        print("-" * 70)

        comparison_count += 1

        if comparison_count >= 5:
            break

Query 5
Query: 혹시나 묵으실 생각 있으신분들은 주소를 잘 검색해서 찾아가세요.
Gold target: 머무르고 싶다면, 주소를 검색해서 찾으십시오.
Pretrained Top-1: 조미료를 쓰는 요리를 해드실 분들은 참고하세요.
Pretrained similarity: 0.8804
Fine-tuned Top-1: 머무르고 싶다면, 주소를 검색해서 찾으십시오.
Fine-tuned similarity: 0.8097
----------------------------------------------------------------------
Query 7
Query: 깨끗하고 와이파이랑 넷플릭스도 됩니다!
Gold target: 깨끗하고 Wi-Fi와 Netflix를 사용할 수 있습니다!
Pretrained Top-1: 화장실은 깨끗하고 따뜻해요!
Pretrained similarity: 0.8593
Fine-tuned Top-1: 깨끗하고 Wi-Fi와 Netflix를 사용할 수 있습니다!
Fine-tuned similarity: 0.8781
----------------------------------------------------------------------
Query 8
Query: 불평할 것 하나 없이 완벽한 곳이었습니다.
Gold target: 그곳은 불평하기에 완벽한 장소였어요.
Pretrained Top-1: 그 외에는, 완벽한 숙박시설이었습니다.
Pretrained similarity: 0.9068
Fine-tuned Top-1: 그곳은 불평하기에 완벽한 장소였어요.
Fine-tuned similarity: 0.8123
----------------------------------------------------------------------
Query 14
Query: 제가 있던 방은 욕실이 많이 좁았어요.
Gold target: 내 방의 화장실은 매우 작았습니다.
Pretrained Top-1: 그 방은 두 사람이 쓰기에는 조금 

In [77]:
pretrained_correct = 0
fine_tuned_correct = 0
corrected_by_fine_tuning = 0
lost_after_fine_tuning = 0

for i in range(len(positive_queries)):
    gold_index = positive_gold_indices[i]

    pretrained_is_correct = (
        pretrained_retrieved_indices[i][0]
        == gold_index
    )

    fine_tuned_is_correct = (
        fine_tuned_retrieved_indices[i][0]
        == gold_index
    )

    if pretrained_is_correct:
        pretrained_correct += 1

    if fine_tuned_is_correct:
        fine_tuned_correct += 1

    if not pretrained_is_correct and fine_tuned_is_correct:
        corrected_by_fine_tuning += 1

    if pretrained_is_correct and not fine_tuned_is_correct:
        lost_after_fine_tuning += 1

print("Top-1 Retrieval Outcome Summary")
print("Pretrained correct:", pretrained_correct)
print("Fine-tuned correct:", fine_tuned_correct)
print(
    "Corrected by fine-tuning:",
    corrected_by_fine_tuning
)
print(
    "Lost after fine-tuning:",
    lost_after_fine_tuning
)

Top-1 Retrieval Outcome Summary
Pretrained correct: 161
Fine-tuned correct: 209
Corrected by fine-tuning: 48
Lost after fine-tuning: 0


# Day 6 Experiment Log

Fine-tuning improved filtered Recall@1 from 0.7318 to 0.9500 under the same FAISS evaluation setup.


# Day 7 - Retrieval Pipeline Modularization and Candidate Generation

Save reusable retriever artifacts and generate candidates for Week 3.


In [78]:
required_variables = [
    "model", "tokenizer", "device",
    "positive_queries", "corpus",
    "positive_gold_indices", "top_k"
]
missing_variables = [
    name for name in required_variables
    if name not in globals()
]
if missing_variables:
    raise NameError(
        "Run Days 1-6 first. Missing variables: "
        f"{missing_variables}"
    )

print("Week 2 Day 7 setup is ready.")
print(f"Queries: {len(positive_queries)} | Corpus: {len(corpus)} | Top-K: {top_k}")


Week 2 Day 7 setup is ready.
Queries: 220 | Corpus: 519 | Top-K: 10


In [79]:
class SentenceEncoder:

    def __init__(
        self,
        model,
        tokenizer,
        device,
        max_length=64,
        batch_size=32
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length
        self.batch_size = batch_size

    def encode(self, sentences):
        self.model.eval()
        all_embeddings = []

        with torch.no_grad():
            for start_idx in range(
                0,
                len(sentences),
                self.batch_size
            ):
                batch_sentences = sentences[
                    start_idx:start_idx + self.batch_size
                ]

                encoded = self.tokenizer(
                    batch_sentences,
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                )

                encoded = {
                    key: value.to(self.device)
                    for key, value in encoded.items()
                }

                embeddings = self.model.encode(
                    encoded["input_ids"],
                    encoded["attention_mask"]
                )

                embeddings = F.normalize(
                    embeddings,
                    p=2,
                    dim=1
                )

                all_embeddings.append(
                    embeddings.cpu().numpy()
                )

        return np.concatenate(
            all_embeddings,
            axis=0
        ).astype("float32")


sentence_encoder = SentenceEncoder(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=64,
    batch_size=32
)

In [80]:
class FaissRetriever:

    def __init__(self, embedding_dimension):
        self.embedding_dimension = embedding_dimension
        self.index = faiss.IndexFlatIP(
            embedding_dimension
        )

    def add(self, corpus_embeddings):
        self.index.add(corpus_embeddings)

    def search(self, query_embeddings, top_k=10):
        similarities, indices = self.index.search(
            query_embeddings,
            top_k
        )

        return similarities, indices

    @property
    def size(self):
        return self.index.ntotal

retriever = FaissRetriever(
    embedding_dimension=768
)

corpus_embeddings = sentence_encoder.encode(
    corpus
)

retriever.add(
    corpus_embeddings
)

print("Embedding dimension:", retriever.embedding_dimension)
print("Indexed corpus vectors:", retriever.size)

Embedding dimension: 768
Indexed corpus vectors: 519


In [81]:
query_embeddings = sentence_encoder.encode(
    positive_queries
)

retrieved_similarities, retrieved_indices = (
    retriever.search(
        query_embeddings,
        top_k=top_k
    )
)

print("Retrieval completed")
print("Query embedding shape:", query_embeddings.shape)
print("Similarity shape:", retrieved_similarities.shape)
print("Retrieved index shape:", retrieved_indices.shape)
print("First query Top-K indices:", retrieved_indices[0])
print(
    "First query Top-K similarities:",
    retrieved_similarities[0]
)

Retrieval completed
Query embedding shape: (220, 768)
Similarity shape: (220, 10)
Retrieved index shape: (220, 10)
First query Top-K indices: [  0  94  34 431 240 370 384 476 156 397]
First query Top-K similarities: [0.9801465  0.93566895 0.9055537  0.87151194 0.8586727  0.77293736
 0.76858616 0.7595434  0.710308   0.7012402 ]


In [82]:
def evaluate_retrieval(
    retrieved_indices,
    gold_indices,
    k_values=(1, 5, 10)
):
    metrics = {}

    for k in k_values:
        hits = []

        for retrieved, gold in zip(
            retrieved_indices,
            gold_indices
        ):
            hit = int(
                gold in retrieved[:k]
            )
            hits.append(hit)

        metrics[f"Recall@{k}"] = (
            np.mean(hits)
        )

    reciprocal_ranks = []

    for retrieved, gold in zip(
        retrieved_indices,
        gold_indices
    ):
        matches = np.where(
            retrieved == gold
        )[0]

        if len(matches) > 0:
            rank = matches[0] + 1
            reciprocal_ranks.append(
                1.0 / rank
            )
        else:
            reciprocal_ranks.append(0.0)

    metrics["MRR@10"] = np.mean(
        reciprocal_ranks
    )

    return metrics


retrieval_metrics = evaluate_retrieval(
    retrieved_indices=retrieved_indices,
    gold_indices=positive_gold_indices,
    k_values=(1, 5, 10)
)

print("Automated Retrieval Evaluation")

for metric, value in retrieval_metrics.items():
    print(f"{metric}: {value:.4f}")

Automated Retrieval Evaluation
Recall@1: 0.9500
Recall@5: 0.9955
Recall@10: 0.9955
MRR@10: 0.9720


In [83]:
def measure_retrieval_latency(
    retriever,
    query_embeddings,
    top_k=10,
    num_runs=10
):
    start_time = time.perf_counter()

    for _ in range(num_runs):
        retriever.search(
            query_embeddings,
            top_k=top_k
        )

    elapsed_time = (
        time.perf_counter() - start_time
    )

    average_latency_ms = (
        elapsed_time / num_runs
    ) * 1000

    latency_per_query_ms = (
        elapsed_time
        / (num_runs * len(query_embeddings))
    ) * 1000

    return {
        "average_latency_ms": average_latency_ms,
        "latency_per_query_ms": latency_per_query_ms
    }


latency_results = measure_retrieval_latency(
    retriever=retriever,
    query_embeddings=query_embeddings,
    top_k=top_k,
    num_runs=10
)

print("Automated Retrieval Latency")
print(
    f"Average search latency: "
    f"{latency_results['average_latency_ms']:.4f} ms"
)
print(
    f"Latency per query: "
    f"{latency_results['latency_per_query_ms']:.6f} ms"
)

Automated Retrieval Latency
Average search latency: 1.6915 ms
Latency per query: 0.007689 ms


In [84]:
def generate_candidates(
    queries,
    corpus,
    retrieved_indices,
    retrieved_similarities
):
    candidates = []

    for query_idx, query in enumerate(queries):
        for rank, (
            corpus_index,
            similarity
        ) in enumerate(
            zip(
                retrieved_indices[query_idx],
                retrieved_similarities[query_idx]
            ),
            start=1
        ):
            candidates.append(
                {
                    "query_id": query_idx,
                    "query": query,
                    "rank": rank,
                    "corpus_index": int(corpus_index),
                    "similarity": float(similarity),
                    "context": corpus[corpus_index]
                }
            )

    return pd.DataFrame(candidates)


candidate_df = generate_candidates(
    queries=positive_queries,
    corpus=corpus,
    retrieved_indices=retrieved_indices,
    retrieved_similarities=retrieved_similarities
)

print("Candidate Generation Completed")
print("Number of candidate rows:", len(candidate_df))
print("Number of queries:", candidate_df["query_id"].nunique())
print("Candidates per query:", top_k)

display(
    candidate_df.head(10)
)

Candidate Generation Completed
Number of candidate rows: 2200
Number of queries: 220
Candidates per query: 10


,query_id,query,rank,corpus_index,similarity,context
0,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,1,0,0.980147,"무엇보다도, 호스트들은 매우 친절했습니다."
1,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,2,94,0.935669,"그리고 무엇보다도, 주인은 매우 친절했어요!"
2,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,3,34,0.905554,그리고 호스트들도 매우 친절했습니다.
3,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,4,431,0.871512,그리고 주인은 매우 친절하고 친절해요!
4,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,5,240,0.858673,특히 사회자는 매우 친절하고 친절합니다.
5,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,6,370,0.772937,호스트 분 답변도 빠르시고 너무 친절하십니다!
6,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,7,384,0.768586,손님에게 매우 친절하고 반응도 좋습니다.
7,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,8,476,0.759543,"우선, 당신의 호스트 엄마가 당신을 만나러 왔고 당신은 매우 친절합니다."
8,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,9,156,0.710308,집은 아주 깔끔하고 호스트도 친절하세요.
9,0,무엇보다도 호스트분들이 너무 친절하셨습니다.,10,397,0.701240,주인은 손도 대지 않고 매우 친절합니다.


In [85]:
sts_candidate_path = (
    "/kaggle/working/week2_sts_retrieval_candidates.csv"
)
candidate_df.to_csv(
    sts_candidate_path,
    index=False,
    encoding="utf-8-sig"
)
print("STS diagnostic candidates saved to:", sts_candidate_path)


STS diagnostic candidates saved to: /kaggle/working/week2_sts_retrieval_candidates.csv


In [86]:
korquad = load_dataset("KorQuAD/squad_kor_v1")

korquad_records = [
    {
        "id": example["id"],
        "title": example["title"],
        "question": example["question"],
        "context": example["context"],
        "answer_text": example["answers"]["text"][0],
    }
    for example in korquad["validation"]
]
korquad_query_df = (
    pd.DataFrame(korquad_records)
    .sample(n=min(1000, len(korquad_records)), random_state=SEED)
    .reset_index(drop=True)
)
korquad_corpus_df = (
    korquad_query_df[["title", "context"]]
    .drop_duplicates(subset=["context"])
    .reset_index(drop=True)
)
context_to_index = {
    context: index
    for index, context in enumerate(korquad_corpus_df["context"])
}
korquad_query_df["gold_index"] = korquad_query_df["context"].map(context_to_index)

korquad_query_embeddings = sentence_encoder.encode(
    korquad_query_df["question"].tolist()
)
korquad_corpus_embeddings = sentence_encoder.encode(
    korquad_corpus_df["context"].tolist()
)
korquad_index = faiss.IndexFlatIP(korquad_corpus_embeddings.shape[1])
korquad_index.add(korquad_corpus_embeddings)
korquad_similarities, korquad_indices = korquad_index.search(
    korquad_query_embeddings,
    20
)

korquad_candidate_records = []
korquad_ranks = []
for query_index, query_row in korquad_query_df.iterrows():
    gold_index = int(query_row["gold_index"])
    retrieved = korquad_indices[query_index]
    gold_positions = np.where(retrieved == gold_index)[0]
    gold_rank = int(gold_positions[0]) + 1 if len(gold_positions) else None
    korquad_ranks.append(gold_rank)
    for rank, (corpus_index, similarity) in enumerate(
        zip(retrieved, korquad_similarities[query_index]),
        start=1
    ):
        corpus_index = int(corpus_index)
        context = korquad_corpus_df.iloc[corpus_index]
        korquad_candidate_records.append({
            "query_id": query_row["id"],
            "query": query_row["question"],
            "question": query_row["question"],
            "title": context["title"],
            "context": context["context"],
            "corpus_index": corpus_index,
            "rank": rank,
            "dense_similarity": float(similarity),
            "label": int(corpus_index == gold_index),
            "gold_in_top20": gold_rank is not None,
            "negative_type": "positive" if corpus_index == gold_index else "dense_candidate"
        })

korquad_candidate_df = pd.DataFrame(korquad_candidate_records)
korquad_retrieval_metrics = {
    "Recall@1": float(np.mean([rank is not None and rank <= 1 for rank in korquad_ranks])),
    "Recall@5": float(np.mean([rank is not None and rank <= 5 for rank in korquad_ranks])),
    "Recall@10": float(np.mean([rank is not None and rank <= 10 for rank in korquad_ranks])),
    "Recall@20": float(np.mean([rank is not None and rank <= 20 for rank in korquad_ranks])),
    "MRR@20": float(np.mean([1.0 / rank if rank is not None else 0.0 for rank in korquad_ranks]))
}

korquad_candidate_path = (
    "/kaggle/working/week2_day7_query_context_candidates.csv"
)
korquad_candidate_df.to_csv(
    korquad_candidate_path,
    index=False,
    encoding="utf-8-sig"
)
print("KorQuAD handoff candidates:", len(korquad_candidate_df))
print("KorQuAD corpus size:", len(korquad_corpus_df))
print("KorQuAD retrieval metrics:", korquad_retrieval_metrics)


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

KorQuAD handoff candidates: 20000
KorQuAD corpus size: 627
KorQuAD retrieval metrics: {'Recall@1': 0.493, 'Recall@5': 0.702, 'Recall@10': 0.788, 'Recall@20': 0.856, 'MRR@20': 0.5878335975256254}


In [87]:
import os
import json
import shutil

artifact_dir = "/kaggle/working/week2_retriever_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

model_checkpoint_path = os.path.join(artifact_dir, "week2_dense_retriever.pt")
torch.save(
    {
        "model_name": model_name,
        "model_state_dict": model.state_dict(),
        "embedding_dimension": int(corpus_embeddings.shape[1]),
        "max_length": MAX_LENGTH,
        "batch_size": 32,
        "training_type": "hard_negative_fine_tuning",
        "seed": int(best_seed),
        "training_seeds": SEEDS
    },
    model_checkpoint_path
)
tokenizer_dir = os.path.join(artifact_dir, "tokenizer")
tokenizer.save_pretrained(tokenizer_dir)

retrieval_config = {
    "model_name": model_name,
    "embedding_dimension": int(corpus_embeddings.shape[1]),
    "max_length": MAX_LENGTH,
    "top_k": top_k,
    "retrieval_score_threshold": RETRIEVAL_SCORE_THRESHOLD,
    "query_count": len(positive_queries),
    "corpus_size": len(corpus),
    "index_type": "IndexFlatIP",
    "seed": int(best_seed),
    "training_seeds": SEEDS
}
with open(os.path.join(artifact_dir, "retrieval_config.json"), "w", encoding="utf-8") as file:
    json.dump(retrieval_config, file, ensure_ascii=False, indent=2)

retrieval_data = {
    "positive_queries": positive_queries,
    "corpus": corpus,
    "positive_gold_indices": [int(index) for index in positive_gold_indices]
}
with open(os.path.join(artifact_dir, "retrieval_data.json"), "w", encoding="utf-8") as file:
    json.dump(retrieval_data, file, ensure_ascii=False, indent=2)

metrics = {
    "sts_baseline": {"pearson": float(pearson), "spearman": float(spearman)},
    "sts_fine_tuned": {"pearson": float(pearson_score), "spearman": float(spearman_score)},
    "filtered_retrieval": {
        "Recall@1": float(positive_recall_at_1),
        "Recall@5": float(positive_recall_at_5),
        "Recall@10": float(positive_recall_at_10),
        "MRR@10": float(retrieval_metrics.get("MRR@10", np.nan))
    },
    "pretrained_retrieval": {key: float(value) for key, value in pretrained_metrics.items()},
    "fine_tuned_retrieval": {key: float(value) for key, value in fine_tuned_metrics.items()},
    "korquad_handoff": korquad_retrieval_metrics,
    "multi_seed_final_training": multi_seed_results_df.to_dict(orient="records")
}
with open(os.path.join(artifact_dir, "week2_metrics.json"), "w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2, allow_nan=False)

multi_seed_results_df.to_csv(os.path.join(artifact_dir, "week2_multi_seed_results.csv"), index=False)
candidate_df.to_csv(os.path.join(artifact_dir, "week2_sts_retrieval_candidates.csv"), index=False, encoding="utf-8-sig")
korquad_candidate_df.to_csv(os.path.join(artifact_dir, "week2_day7_query_context_candidates.csv"), index=False, encoding="utf-8-sig")
korquad_retrieval_data = {
    "queries": korquad_query_df["question"].tolist(),
    "corpus": korquad_corpus_df["context"].tolist(),
    "gold_indices": [int(index) for index in korquad_query_df["gold_index"]]
}
with open(os.path.join(artifact_dir, "korquad_retrieval_data.json"), "w", encoding="utf-8") as file:
    json.dump(korquad_retrieval_data, file, ensure_ascii=False, indent=2)

reloaded_checkpoint = torch.load(model_checkpoint_path, map_location=device)
reloaded_model = RobertaSTSModel(reloaded_checkpoint["model_name"]).to(device)
reloaded_model.load_state_dict(reloaded_checkpoint["model_state_dict"])
reloaded_model.eval()
reloaded_encoder = SentenceEncoder(reloaded_model, tokenizer, device, max_length=MAX_LENGTH, batch_size=32)
reloaded_embeddings = reloaded_encoder.encode(positive_queries[:2])
original_embeddings = encode_sentences_with_model(positive_queries[:2], model, tokenizer, device)
assert np.allclose(original_embeddings, reloaded_embeddings, atol=1e-5)
print("Artifact round-trip check: passed")

zip_path = shutil.make_archive(
    "/kaggle/working/week2_retriever_artifacts",
    "zip",
    artifact_dir
)
print("Week 2 artifacts saved:", artifact_dir)
print("ZIP path:", zip_path)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Artifact round-trip check: passed
Week 2 artifacts saved: /kaggle/working/week2_retriever_artifacts
ZIP path: /kaggle/working/week2_retriever_artifacts.zip


# Day 7 Experiment Log

The saved retriever reached KorQuAD Recall@20 0.8560 and MRR@20 0.5878. Its checkpoint, tokenizer, configuration, and Top-20 candidates were passed to Week 3.
